# Lecture 19 — Simulation-Based Inference: Neural Posterior Estimation

**PHYG004 / PHY5006, 2026 Spring · Sogang University**
Prof. Young Woo Choi

---

## Where we are in the course

We just finished the **generative-models block**:

| Lecture | Model | What it gives you |
|---|---|---|
| L14 | VAE | latent variables, approximate density |
| L15 | **Normalizing flow (RealNVP)** | *exact* density $p_\theta(x)$, one-shot samples |
| L16 | Diffusion | high-quality samples |
| L17–18 | Score / flow matching | continuous-time generative dynamics |

Today opens the **inference & frontier block**. The single most useful thing a flow can
do for a physicist is not "make pretty samples" — it is to *answer an inverse question*:

> *Given some data I measured, what were the underlying physical parameters that produced it,
> and how uncertain am I about them?*

That is **Bayesian inference**, and the modern neural way to do it is called
**simulation-based inference (SBI)**, a.k.a. *likelihood-free inference*. Today we build
the flagship SBI method — **Neural Posterior Estimation (NPE)** — by taking the **exact
same RealNVP code from Lecture 15** and making one small change: we let the flow's neural
networks *peek at the observation*. That turns a generative flow into a **conditional**
flow that directly models the posterior $p(\theta \mid x_{\text{obs}})$.

## What you'll learn today

1. State the inverse problem in Bayesian language: prior, likelihood, **posterior**.
2. Understand why the likelihood is often **intractable** for a physics simulator, and why that breaks classical inference.
3. **Extend L15's `AffineCoupling` into a `ConditionalAffineCoupling`** in ~15 lines — the only new code.
4. Train an **amortized** posterior estimator: one training run, then *instant* posteriors for any future observation.
5. Apply it to two physics simulators: a noisy **harmonic-oscillator** trajectory (infer $\omega^2$) and a **double-well** Metropolis chain (infer temperature).
6. Compare NPE against the **L08 PINN inverse problem** and **L09 differentiable-physics** gradient-based calibration.
7. Validate the posterior with a **coverage check (SBC-lite)** — does "90% credible" really mean 90%?

> **Runtime tip.** Everything runs on the free Colab CPU/T4 in well under 10 minutes. No downloads, no real datasets — every "observation" comes from a simulator we write ourselves.

> **Prerequisites.** L15 (RealNVP coupling layers, `flow.log_prob`, the change-of-variables loss). A working memory of "prior × likelihood ∝ posterior" from any stats course is enough; we re-derive what we need.


## 0. Setup

In [ ]:
# Colab: uncomment to install. The core stack is the same as Lecture 15.
# !pip install -q jax jaxlib optax flax matplotlib
#
# Optional: the `sbi` package (PyTorch-based) is used ONLY in the closing survey
# section to point at production tools. The whole notebook runs without it.
# !pip install -q sbi

import jax
import jax.numpy as jnp
import jax.random as jr
import optax
import flax.nnx as nnx
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from functools import partial

KEY = jr.PRNGKey(42)          # course-wide default seed
print(f"JAX backend: {jax.default_backend()}")


## 1. The inverse problem, in one picture

Every physics simulation is a **forward map**:

$$
\theta \;\xrightarrow{\;\text{simulator}\;}\; x
\qquad\text{(parameters} \to \text{data)}.
$$

You set the coupling constant / temperature / mass $\theta$, run the dynamics, and out
comes an observable $x$ (a trajectory, a spectrum, a histogram). Forward is easy: it is
just running the code.

What we actually *want in the lab* is the **inverse**:

$$
x_{\text{obs}} \;\xrightarrow{\;?\;}\; \theta
\qquad\text{(data} \to \text{parameters)}.
$$

We measured $x_{\text{obs}}$; which $\theta$ produced it? Because data are noisy and the
map can be many-to-one, the honest answer is **not a single number** but a **probability
distribution** — the **posterior** $p(\theta \mid x_{\text{obs}})$. Its width *is* your
error bar.

### Bayes' theorem — the object we are after

$$
\underbrace{p(\theta \mid x_{\text{obs}})}_{\text{posterior}}
\;=\;
\frac{\overbrace{p(x_{\text{obs}} \mid \theta)}^{\text{likelihood}}\;\overbrace{p(\theta)}^{\text{prior}}}
{\underbrace{p(x_{\text{obs}})}_{\text{evidence}}}.
$$

- **Prior** $p(\theta)$ — what we believed about $\theta$ *before* measuring (e.g. "$\omega^2$ is somewhere in $[0.5, 2.0]$").
- **Likelihood** $p(x \mid \theta)$ — how probable the data are for a given $\theta$. This is where the physics lives.
- **Posterior** — belief *after* seeing the data. This is the deliverable.

### Why this is hard: the likelihood is intractable

For most real simulators you can *run* the forward map but you **cannot write down**
$p(x \mid \theta)$ as a formula. A Langevin trajectory, a Monte-Carlo histogram, a
detector response — each is a stochastic black box. There is no closed-form density to
plug into Bayes' theorem, and no way to evaluate the likelihood pointwise.

> This is the **likelihood-free** setting. Classical tools (MCMC, gradient descent on a
> log-likelihood) need an *evaluable* likelihood, so they stall here. We need a method
> that learns from *samples of the forward map alone* — which is exactly what a
> normalizing flow can do.


In [ ]:
# A cartoon of forward (easy) vs inverse (what we want).
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.set_xlim(0, 11); ax.set_ylim(0, 3.2); ax.axis("off")

def box(x, y, w, h, label, fc, ec="#37474f"):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.06",
                                 fc=fc, ec=ec, lw=1.3))
    ax.text(x + w/2, y + h/2, label, ha="center", va="center", fontsize=11)

def arr(x1, y1, x2, y2, color, label="", up=True):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="-|>", color=color, lw=2.0))
    if label:
        ax.text((x1+x2)/2, (y1+y2)/2 + (0.28 if up else -0.34), label,
                ha="center", fontsize=10, color=color)

box(0.4, 1.2, 2.4, 0.9, r"parameters $\theta$" + "\n($\\omega^2$, $k_BT$, ...)", "#e3f2fd")
box(4.2, 1.2, 2.6, 0.9, "simulator\n(forward map)", "#fff3e0")
box(8.2, 1.2, 2.4, 0.9, r"data $x_{\rm obs}$" + "\n(trajectory, histogram)", "#fce4ec")

arr(2.8, 1.85, 4.2, 1.85, "#2e7d32", "easy:  run the code", up=True)
arr(6.8, 1.85, 8.2, 1.85, "#2e7d32", "", up=True)
arr(8.2, 1.45, 2.8, 1.45, "#c62828", "hard:  posterior  $p(\\theta\\,|\\,x_{\\rm obs})$", up=False)

plt.tight_layout(); plt.show()


## 2. Three ways to invert — and where SBI fits

We have already met two inverse-problem strategies earlier in the course. Today's NPE is a
third, with a very different cost/benefit profile. It is worth seeing all three side by side
*before* writing code, so you know what NPE buys you.

| | **L08 — PINN inverse** | **L09 — differentiable physics** | **L19 — NPE (today)** |
|---|---|---|---|
| What you optimize | physics residual + data fit, w.r.t. $\theta$ | data-misfit loss, backprop *through the simulator* | a conditional flow's NLL, w.r.t. flow weights |
| Needs a differentiable simulator? | the PDE is encoded in the loss | **yes** — gradients flow through the solver | **no** — only forward samples $(\theta, x)$ |
| Needs an evaluable likelihood? | implicit (Gaussian data term) | implicit (squared misfit) | **no** — likelihood-free |
| Output | a **point estimate** $\hat\theta$ | a **point estimate** $\hat\theta$ | a **full posterior** $p(\theta\mid x)$ |
| Uncertainty quantification | not native (needs extra work) | not native | **built in** — the posterior *is* the error bar |
| Cost per *new* observation | re-run the whole optimization | re-run the whole optimization | **one forward pass (~ms)** — *amortized* |
| Upfront cost | none (just solve) | none | train once on many simulations |

Read the **last two rows** carefully — they are the whole point of SBI:

- **Amortization.** L08/L09 solve a fresh optimization problem for *every* new measurement.
  NPE pays a one-time training cost, then answers any future $x_{\text{obs}}$ with a single
  network evaluation. If you analyze thousands of events (gravitational-wave catalogs,
  particle collisions, repeated lab runs), this is transformative.
- **Native uncertainty.** L08/L09 hand you a best-fit number. NPE hands you the *distribution*,
  so "$\omega^2 = 1.0 \pm 0.1$" comes out for free, with calibration you can actually check
  (Section 8). This is the UQ thread the course keeps returning to.

The price: NPE needs an upfront set of simulations and only learns the posterior on the
*family* of observations it was trained on. For a one-off fit of a single dataset, a PINN or a
differentiable solver may well be simpler. **Match the tool to the job.**


## 3. Recall the Lecture 15 flow (the unconditional version)

Before we make it conditional, let's drop in the **exact** RealNVP from L15 so the notebook
is self-contained. Nothing here is new — read it as a refresher.

Recall the **coupling layer**: split a vector into a *kept* half $z_A$ and a *transformed*
half $z_B$, and set

$$
x_A = z_A, \qquad x_B = z_B \cdot e^{\,s(z_A)} + t(z_A),
$$

where $s, t$ are small neural networks. The Jacobian is triangular, so
$\log|\det J| = \sum_i s_i(z_A)$ is cheap, and the map is trivially invertible. A
`RealNVP` stacks several of these with alternating masks and exposes

$$
\log p_\theta(x) \;=\; \log \mathcal{N}\!\big(f^{-1}(x);\,0,I\big) \;+\; \log\big|\det J_{f^{-1}}(x)\big|.
$$


In [ ]:
# ── Lecture 15 RealNVP, verbatim (unconditional). Skim; the next section extends it. ──
class AffineCoupling(nnx.Module):
    # mask[i] == 1  -> dimension i is "kept";  mask[i] == 0  -> "transformed".
    def __init__(self, d, hidden, mask, *, rngs):
        self.mask = mask
        self.net = nnx.Sequential(
            nnx.Linear(d, hidden, rngs=rngs), nnx.relu,
            nnx.Linear(hidden, hidden, rngs=rngs), nnx.relu,
            nnx.Linear(hidden, 2 * d, rngs=rngs),   # outputs s and t stacked
        )
        self.d = d

    def _scale_and_shift(self, kept):
        st = self.net(kept)
        s_raw, t = st[..., :self.d], st[..., self.d:]
        s = jnp.tanh(s_raw) * 2.0                   # bound |s| < 2 for stability
        return s, t

    def forward(self, z):
        s, t = self._scale_and_shift(z * self.mask)
        transformed = 1.0 - self.mask
        x = z * jnp.exp(s * transformed) + t * transformed
        log_det = jnp.sum(s * transformed, axis=-1)
        return x, log_det

    def inverse(self, x):
        s, t = self._scale_and_shift(x * self.mask)
        transformed = 1.0 - self.mask
        z = (x - t * transformed) * jnp.exp(-s * transformed)
        log_det = -jnp.sum(s * transformed, axis=-1)
        return z, log_det


class RealNVP(nnx.Module):
    def __init__(self, d=2, n_layers=8, hidden=64, *, rngs):
        masks = []
        for i in range(n_layers):
            mask = jnp.zeros(d)
            mask = mask.at[:d//2].set(1.0) if i % 2 == 0 else mask.at[d//2:].set(1.0)
            masks.append(mask)
        self.layers = nnx.List([AffineCoupling(d, hidden, m, rngs=rngs) for m in masks])
        self.d = d

    def forward(self, z):
        log_det = jnp.zeros(z.shape[0]); x = z
        for layer in self.layers:
            x, ld = layer.forward(x); log_det += ld
        return x, log_det

    def inverse(self, x):
        log_det = jnp.zeros(x.shape[0]); z = x
        for layer in reversed(self.layers):
            z, ld = layer.inverse(z); log_det += ld
        return z, log_det

    def log_prob(self, x):
        z, log_det_inv = self.inverse(x)
        log_pz = -0.5 * jnp.sum(z**2, axis=-1) - 0.5 * self.d * jnp.log(2 * jnp.pi)
        return log_pz + log_det_inv

    def sample(self, key, n):
        z = jr.normal(key, (n, self.d))
        x, _ = self.forward(z)
        return x

print("RealNVP (L15) loaded.")


## 4. The one new idea — make the flow *conditional*

Here is the entire conceptual leap of today's lecture, in two lines.

An **unconditional** flow models a single fixed density $p_\theta(x)$. We want a *family* of
densities, **one posterior per observation**:

$$
p_\theta(\theta_{\text{phys}} \mid x_{\text{obs}}).
$$

(From here on, to avoid clashing with the flow's own weights, the **physical parameter we
infer** is written $\vartheta$ and the **observation** is $c$ — "the condition".)

To turn the L15 flow into a conditional one, we feed the observation $c$ into the coupling
networks as an extra input. The single new line is in `_scale_and_shift`:

```python
# L15:                  st = self.net(z * mask)
# L19 (conditional):    st = self.net( jnp.concatenate([z * mask, c], axis=-1) )
```

That's it. The scale/shift now depend on *both* the kept dimensions **and** the observation,
so the learned transformation — and hence the modelled density over $\vartheta$ — *changes
with the data*. Everything else (triangular Jacobian, exact `log_prob`, invertibility) is
untouched, because $c$ is held fixed during the transform: it is a *side input*, not a
variable being transformed, so it never enters the Jacobian.

> **Physics analogy.** Think of $c$ as an *external field*. The unconditional flow samples a
> fixed Hamiltonian; the conditional flow samples a Hamiltonian whose couplings are *tuned by
> the measured data*. Same machinery, an extra knob wired to the observation.


In [ ]:
# Diagram: where the observation c enters the coupling layer.
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.set_xlim(0, 10); ax.set_ylim(0, 3.4); ax.axis("off")

def box(x, y, w, h, label, fc, ec="#37474f"):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.06",
                                 fc=fc, ec=ec, lw=1.3))
    ax.text(x + w/2, y + h/2, label, ha="center", va="center", fontsize=11)

def arr(x1, y1, x2, y2, color="#37474f"):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="-|>", color=color, lw=1.4))

box(0.2, 2.3, 1.4, 0.9, r"$\vartheta_A$" + "\n(kept)", "#e3f2fd")
box(0.2, 1.2, 1.4, 0.9, r"$\vartheta_B$", "#fce4ec")
box(0.2, 0.1, 1.4, 0.9, r"obs $c$", "#e8f5e9", ec="#2e7d32")
box(3.4, 1.4, 2.0, 1.2, r"$s, t$ net" + "\ninput = $[\\vartheta_A,\\,c]$", "#fff3e0")
box(6.6, 1.2, 2.2, 0.9, r"$\vartheta_B e^{s} + t$", "#fce4ec", ec="#c2185b")

arr(1.6, 2.75, 3.4, 2.3)        # theta_A -> net
arr(1.6, 0.55, 3.4, 1.7)        # c -> net
arr(1.6, 1.65, 6.6, 1.55)       # theta_B -> scale&shift
arr(5.4, 1.9, 6.6, 1.7)         # net -> scale&shift
ax.text(4.4, 0.3, r"$c$ enters the net but is NOT transformed $\Rightarrow$ Jacobian unchanged",
        ha="center", fontsize=9.5, color="#2e7d32")
plt.tight_layout(); plt.show()


### Checkpoint 1 — implement the conditional coupling layer

Your first task: complete `ConditionalAffineCoupling` below. It is `AffineCoupling` with two
edits, both marked `# TODO`:

1. The `net` must accept the concatenated input `[kept_dims, c]`, so its input width is `d + c_dim` (not `d`).
2. In `_scale_and_shift`, concatenate `c` onto the kept dimensions before calling `net`.

**Verification criteria (run the check cell after):**
- `flow.log_prob(theta, c)` returns shape `(batch,)`.
- Varying `c` with `theta` fixed *changes* `log_prob` (the condition actually does something).


In [ ]:
class ConditionalAffineCoupling(nnx.Module):
    '''AffineCoupling (L15) extended so s, t also depend on a condition c.'''
    def __init__(self, d, c_dim, hidden, mask, *, rngs):
        self.mask = mask
        self.d = d
        self.c_dim = c_dim
        # TODO (Checkpoint 1, edit 1):
        #   The net now sees [kept_dims (size d), condition c (size c_dim)].
        #   Set the FIRST Linear's input width to (d + c_dim) instead of d.
        self.net = nnx.Sequential(
            nnx.Linear(d + c_dim, hidden, rngs=rngs), nnx.relu,   # <-- input width = d + c_dim
            nnx.Linear(hidden, hidden, rngs=rngs), nnx.relu,
            nnx.Linear(hidden, 2 * d, rngs=rngs),
        )

    def _scale_and_shift(self, kept, c):
        # TODO (Checkpoint 1, edit 2):
        #   Concatenate the observation c onto the kept dims before the net.
        #   L15 was:  st = self.net(kept)
        net_in = jnp.concatenate([kept, c], axis=-1)            # <-- the one new line
        st = self.net(net_in)
        s_raw, t = st[..., :self.d], st[..., self.d:]
        s = jnp.tanh(s_raw) * 2.0
        return s, t

    def forward(self, z, c):
        s, t = self._scale_and_shift(z * self.mask, c)
        transformed = 1.0 - self.mask
        x = z * jnp.exp(s * transformed) + t * transformed
        log_det = jnp.sum(s * transformed, axis=-1)
        return x, log_det

    def inverse(self, x, c):
        s, t = self._scale_and_shift(x * self.mask, c)
        transformed = 1.0 - self.mask
        z = (x - t * transformed) * jnp.exp(-s * transformed)
        log_det = -jnp.sum(s * transformed, axis=-1)
        return z, log_det


In [ ]:
class ConditionalRealNVP(nnx.Module):
    '''Stack of conditional coupling layers. Models p(theta | c).

    `d`     = dimension of the parameter theta we infer.
    `c_dim` = dimension of the observation / summary statistic c.
    '''
    def __init__(self, d, c_dim, n_layers=8, hidden=64, *, rngs):
        masks = []
        for i in range(n_layers):
            mask = jnp.zeros(d)
            # For d==1 we alternate keep/transform the single dim across layers.
            if d == 1:
                mask = mask.at[0].set(1.0 if i % 2 == 0 else 0.0)
            else:
                mask = mask.at[:d//2].set(1.0) if i % 2 == 0 else mask.at[d//2:].set(1.0)
            masks.append(mask)
        self.layers = nnx.List(
            [ConditionalAffineCoupling(d, c_dim, hidden, m, rngs=rngs) for m in masks]
        )
        self.d = d
        self.c_dim = c_dim

    def forward(self, z, c):
        log_det = jnp.zeros(z.shape[0]); x = z
        for layer in self.layers:
            x, ld = layer.forward(x, c); log_det += ld
        return x, log_det

    def inverse(self, x, c):
        log_det = jnp.zeros(x.shape[0]); z = x
        for layer in reversed(self.layers):
            z, ld = layer.inverse(z, c); log_det += ld
        return z, log_det

    def log_prob(self, theta, c):
        # log p(theta | c) via change of variables, with c held fixed.
        z, log_det_inv = self.inverse(theta, c)
        log_pz = -0.5 * jnp.sum(z**2, axis=-1) - 0.5 * self.d * jnp.log(2 * jnp.pi)
        return log_pz + log_det_inv

    def sample(self, key, c):
        # One posterior sample per row of c. c has shape (n, c_dim).
        n = c.shape[0]
        z = jr.normal(key, (n, self.d))
        theta, _ = self.forward(z, c)
        return theta

    def sample_given_one(self, key, c_single, n):
        # Draw n posterior samples for a SINGLE observation c_single (shape (c_dim,)).
        c_rep = jnp.broadcast_to(c_single, (n, self.c_dim))
        return self.sample(key, c_rep)

print("ConditionalRealNVP defined.")


In [ ]:
# Checkpoint-1 verification: shapes, and that c actually matters.
flow_demo = ConditionalRealNVP(d=1, c_dim=3, n_layers=6, hidden=32, rngs=nnx.Rngs(0))

theta = jr.normal(jr.PRNGKey(1), (5, 1))     # 5 parameter values
c      = jr.normal(jr.PRNGKey(2), (5, 3))    # 5 observations (c_dim = 3)

lp = flow_demo.log_prob(theta, c)
print(f"theta shape: {theta.shape},  c shape: {c.shape}")
print(f"log_prob shape: {lp.shape}   (must be (5,))")
assert lp.shape == (5,), "log_prob must be (batch,)"

# Does the condition do anything? Fix theta, sweep c, watch log_prob move.
theta_fixed = jnp.zeros((20, 1))
c_sweep     = jnp.linspace(-2, 2, 20)[:, None] * jnp.ones((1, 3))
lp_sweep    = flow_demo.log_prob(theta_fixed, c_sweep)
spread = float(lp_sweep.max() - lp_sweep.min())
print(f"log_prob spread as c varies (theta fixed): {spread:.3f}  (should be > 0)")
assert spread > 1e-3, "condition c has no effect -- check the concatenation!"

plt.figure(figsize=(6, 3))
plt.scatter(np.asarray(c_sweep[:, 0]), np.asarray(lp_sweep), c="C0")
plt.xlabel("condition $c$ (swept)"); plt.ylabel(r"$\log p(\vartheta{=}0 \mid c)$")
plt.title("Checkpoint 1: the condition changes the density (not monotone — good)")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print("Checkpoint 1 passed.")


**Read the scatter.** As $c$ varies, $\log p(\vartheta{=}0\mid c)$ wanders up and down —
it is *not* a flat line and *not* monotone. That non-trivial dependence is exactly what we
need: the posterior over $\vartheta$ genuinely reshapes itself in response to the data. (This
flow is untrained, so the pattern is meaningless — we are only checking that the wiring works.)


## 5. Target A — infer a spring constant from a noisy trajectory

Our first physical inference target is the workhorse of all of physics: the **1D harmonic
oscillator** (the same system the L08 PINN handled, now revisited from the *inference* angle).
A particle in a quadratic potential obeys

$$
\ddot{x} = -\omega^2\, x,
$$

so the unknown physical parameter is the **squared frequency** $\vartheta \equiv \omega^2$
(the stiffness of the spring, in units where $m=1$). Our **prior** is a uniform belief
$\omega^2 \sim \mathcal{U}[0.5, 2.0]$.

**The forward simulator.** Given $\omega^2$, we integrate the oscillator from a fixed start,
add measurement noise to each recorded position, and keep a short trajectory of $T=20$ steps.
This *is* the likelihood-free setting: the noisy trajectory has a density we never write down
analytically — we only know how to *sample* it.

**The observation we feed the flow.** A raw 20-step trajectory is a high-dimensional, noisy
object. As is standard in SBI, we compress it to a few **summary statistics** that carry the
information about $\omega^2$: the oscillation looks faster for stiffer springs, so the dominant
period (and the trajectory's variance / zero-crossing rate) are informative. We use a small,
physically-motivated summary vector.


In [ ]:
# ── Forward simulator A: noisy harmonic-oscillator trajectory ──
T_STEPS = 20
DT = 0.3

@partial(jax.jit, static_argnames=("n_steps",))
def simulate_oscillator(omega2, key, n_steps=T_STEPS, dt=DT, x0=1.0, v0=0.0, noise=0.05):
    '''Leapfrog-integrate xddot = -omega2 x, return a noisy position trajectory.

    omega2 : scalar ()       physical parameter (spring stiffness)
    returns: (n_steps,) array of noisy positions.
    '''
    omega2 = jnp.asarray(omega2)
    def step(carry, k):
        x, v = carry
        v = v - omega2 * x * (dt / 2)      # half kick
        x = x + v * dt                     # drift
        v = v - omega2 * x * (dt / 2)      # half kick
        x_obs = x + noise * jr.normal(k)   # measurement noise
        return (x, v), x_obs
    keys = jr.split(key, n_steps)
    _, traj = jax.lax.scan(step, (jnp.asarray(x0), jnp.asarray(v0)), keys)
    return traj                            # (n_steps,)


def summary_oscillator(traj):
    '''Compress a trajectory to a 4-D summary statistic (informative about omega2).'''
    var = jnp.var(traj)
    # zero-crossing rate ~ frequency
    signs = jnp.sign(traj)
    crossings = jnp.mean(jnp.abs(jnp.diff(signs)) > 0)
    # lag-1 autocorrelation (faster oscillation -> more negative)
    tc = traj - jnp.mean(traj)
    ac1 = jnp.sum(tc[:-1] * tc[1:]) / (jnp.sum(tc**2) + 1e-8)
    rng = jnp.max(traj) - jnp.min(traj)
    return jnp.stack([var, crossings, ac1, rng])    # (4,)

C_DIM_A = 4

# Sanity check: a stiff spring oscillates faster than a soft one.
key = jr.PRNGKey(0)
traj_soft  = simulate_oscillator(0.6, key)
traj_stiff = simulate_oscillator(1.9, key)
print(f"trajectory shape: {traj_soft.shape}")
print(f"summary(soft  omega2=0.6): {np.asarray(summary_oscillator(traj_soft))}")
print(f"summary(stiff omega2=1.9): {np.asarray(summary_oscillator(traj_stiff))}")

plt.figure(figsize=(8, 3))
plt.plot(np.asarray(traj_soft),  "o-", ms=3, label=r"$\omega^2=0.6$ (soft)")
plt.plot(np.asarray(traj_stiff), "s-", ms=3, label=r"$\omega^2=1.9$ (stiff)")
plt.xlabel("time step"); plt.ylabel("noisy position $x$")
plt.title("Forward simulator A: noisy oscillator trajectories")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


### Build the training set: draw from the prior, run the simulator

The recipe for NPE training data is the **forward process itself**:

$$
\vartheta^{(i)} \sim p(\vartheta), \qquad
x^{(i)} \sim \text{simulator}(\vartheta^{(i)}), \qquad
c^{(i)} = \text{summary}(x^{(i)}),
$$

giving pairs $\{(\vartheta^{(i)}, c^{(i)})\}_{i=1}^N$. We never evaluate a likelihood — we only
*run the forward map*, which we can always do. The flow will learn $p(\vartheta\mid c)$ from
these pairs alone.


In [ ]:
# Standardize parameter and summary for stable training; keep the transforms to invert later.
def make_dataset_A(n, key):
    k_theta, k_sim = jr.split(key)
    # Prior: omega^2 ~ U[0.5, 2.0]
    omega2 = jr.uniform(k_theta, (n,), minval=0.5, maxval=2.0)
    sim_keys = jr.split(k_sim, n)
    def one(o2, k):
        return summary_oscillator(simulate_oscillator(o2, k))
    C = jax.vmap(one)(omega2, sim_keys)            # (n, 4)
    return omega2[:, None], C                       # theta (n,1), c (n,4)

N_TRAIN_A = 8000
theta_A, c_A = make_dataset_A(N_TRAIN_A, jr.PRNGKey(7))
print(f"theta_A shape: {theta_A.shape},  c_A shape: {c_A.shape}")

# Standardizers (computed on the training set; reused for any new observation).
theta_mean, theta_std = theta_A.mean(0), theta_A.std(0)
c_mean, c_std = c_A.mean(0), c_A.std(0) + 1e-6

def norm_theta(t): return (t - theta_mean) / theta_std
def denorm_theta(t): return t * theta_std + theta_mean
def norm_c(c): return (c - c_mean) / c_std

theta_A_n = norm_theta(theta_A)
c_A_n     = norm_c(c_A)
print(f"standardized theta mean/std: {np.asarray(theta_A_n.mean()):.3f} / {np.asarray(theta_A_n.std()):.3f}")


## 6. Train the Neural Posterior Estimator

Now the magic — and the punch line of the whole lecture. The NPE training objective is the
*same maximum-likelihood loss as L15*, just **conditional**:

$$
\mathcal{L}(\phi)
= -\,\mathbb{E}_{(\vartheta, c)\sim\text{simulator}}
   \big[\,\log q_\phi(\vartheta \mid c)\,\big],
$$

where $q_\phi$ is our conditional flow (weights $\phi$). Minimizing it drives $q_\phi(\vartheta\mid c)$
toward the **true posterior** $p(\vartheta\mid c)$. The remarkable fact (Papamakarios &
Murray 2016) is that fitting the flow to forward-simulated *pairs* recovers the Bayesian
posterior — no likelihood evaluation anywhere.

In code, the loss is a one-liner.


### Checkpoint 2 — complete the NPE loss and training loop

Complete `npe_loss` below (one `# TODO` line). It is the conditional negative log-likelihood:
return `-flow.log_prob(theta_batch, c=c_batch).mean()`.

**Physics verification criterion (checked after training):** for a held-out observation
generated with $\omega^2_{\text{true}} = 1.0$, the **posterior mean** should land within
$\pm 0.15$ of $1.0$, and the **posterior std** should be *narrower than the prior std*
(the prior $\mathcal{U}[0.5,2.0]$ has std $\approx 0.433$). Narrower-than-prior means the data
were informative.


In [ ]:
def npe_loss(flow, theta_batch, c_batch):
    # TODO (Checkpoint 2): conditional negative log-likelihood.
    #   Return the mean of -log q(theta | c) over the batch.
    return -flow.log_prob(theta_batch, c_batch).mean()


def train_npe(flow, theta_n, c_n, n_epochs=500, batch_size=256, lr=1e-3, seed=42):
    optimizer = nnx.Optimizer(flow, optax.adam(lr), wrt=nnx.Param)

    @nnx.jit
    def train_step(flow, optimizer, tb, cb):
        loss, grads = nnx.value_and_grad(
            npe_loss, argnums=nnx.DiffState(0, nnx.Param))(flow, tb, cb)
        optimizer.update(flow, grads)
        return loss

    losses, key = [], jr.PRNGKey(seed)
    n = theta_n.shape[0]
    for epoch in range(n_epochs):
        key, sub = jr.split(key)
        perm = jr.permutation(sub, n)
        tn, cn = theta_n[perm], c_n[perm]
        ep, nb = 0.0, 0
        for i in range(0, n, batch_size):
            tb, cb = tn[i:i+batch_size], cn[i:i+batch_size]
            if len(tb) < 2: continue
            ep += float(train_step(flow, optimizer, tb, cb)); nb += 1
        losses.append(ep / max(nb, 1))
        if (epoch + 1) % 100 == 0:
            print(f"epoch {epoch+1:4d}   NLL = {losses[-1]:.4f}")
    return losses


In [ ]:
# Train the NPE for target A. d=1 (omega^2), c_dim=4 (summary).
npe_A = ConditionalRealNVP(d=1, c_dim=C_DIM_A, n_layers=10, hidden=64, rngs=nnx.Rngs(0))
losses_A = train_npe(npe_A, theta_A_n, c_A_n, n_epochs=500, batch_size=256, lr=1e-3)

plt.figure(figsize=(7, 3))
plt.plot(losses_A); plt.xlabel("epoch"); plt.ylabel("NLL")
plt.title("NPE training loss (target A: oscillator)"); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# ── Checkpoint-2 verification: posterior for a held-out omega^2_true = 1.0 ──
omega2_true = 1.0
x_obs = summary_oscillator(simulate_oscillator(omega2_true, jr.PRNGKey(123)))   # (4,)
c_obs_n = norm_c(x_obs[None, :])[0]                                             # (4,) standardized

# Draw posterior samples (in standardized theta space) for this one observation.
post_n = npe_A.sample_given_one(jr.PRNGKey(55), c_obs_n, 4000)                  # (4000, 1)
post   = np.asarray(denorm_theta(post_n))[:, 0]                                 # physical omega^2

post_mean, post_std = post.mean(), post.std()
prior_std = (2.0 - 0.5) / np.sqrt(12)        # std of U[0.5, 2.0] ~ 0.433
print(f"posterior mean = {post_mean:.3f}  (true = {omega2_true})")
print(f"posterior std  = {post_std:.3f}    (prior std = {prior_std:.3f})")
assert abs(post_mean - omega2_true) < 0.15, "posterior mean off by > 0.15"
assert post_std < prior_std, "posterior not narrower than prior -- data should be informative"

plt.figure(figsize=(7, 3.4))
plt.hist(post, bins=50, density=True, color="C0", alpha=0.65, label="posterior $p(\\omega^2|x_{\\rm obs})$")
plt.axvspan(0.5, 2.0, color="gray", alpha=0.12, label="prior $\\mathcal{U}[0.5,2.0]$")
plt.axvline(omega2_true, color="C3", lw=2, label=f"true $\\omega^2={omega2_true}$")
plt.axvline(post_mean, color="C0", ls="--", lw=2, label="posterior mean")
plt.xlabel(r"$\omega^2$"); plt.ylabel("density"); plt.legend(fontsize=8)
plt.title("Checkpoint 2: amortized posterior concentrates near the truth")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print("Checkpoint 2 passed.")


**Read the histogram.** The posterior (blue) is a *tight bump* near $\omega^2 = 1.0$,
much narrower than the flat gray prior. The data pinned down the spring constant, and the
spread of the bump *is* the error bar — no extra bootstrap, no Hessian, no second optimization.
That is the Bayesian deliverable we set out for in Section 1.


## 7. The payoff — *amortized* inference

Here is what separates SBI from L08/L09. The flow is **trained once**. Now hand it a *brand new*
observation it never saw and it returns the full posterior in **one forward pass (~milliseconds)**
— no re-optimization, no MCMC chain.

Below we sweep several true values of $\omega^2$, simulate a fresh noisy trajectory for each,
and read off the posterior instantly. Watch the posterior **track the truth** across the whole
prior range, and time how fast it is.


In [ ]:
import time

omega2_grid = jnp.array([0.6, 0.9, 1.2, 1.5, 1.8])
fig, axes = plt.subplots(1, len(omega2_grid), figsize=(15, 3), sharey=True)

t0 = time.time()
for ax, o2 in zip(axes, omega2_grid):
    x_new   = summary_oscillator(simulate_oscillator(float(o2), jr.PRNGKey(int(o2 * 1000))))
    c_new_n = norm_c(x_new[None, :])[0]
    samp    = np.asarray(denorm_theta(
                  npe_A.sample_given_one(jr.PRNGKey(9), c_new_n, 3000)))[:, 0]
    ax.hist(samp, bins=40, density=True, color="C0", alpha=0.65)
    ax.axvline(float(o2), color="C3", lw=2)
    ax.axvline(samp.mean(), color="C0", ls="--", lw=1.5)
    ax.set_title(f"true $\\omega^2={float(o2):.1f}$", fontsize=10)
    ax.set_xlim(0.4, 2.1); ax.set_xlabel(r"$\omega^2$"); ax.grid(alpha=0.3)
axes[0].set_ylabel("posterior density")
elapsed = time.time() - t0
plt.suptitle(f"Amortized NPE: 5 new observations, full posteriors in {elapsed*1000:.0f} ms total", y=1.04)
plt.tight_layout(); plt.show()

print(f"Average time per new posterior (incl. simulation): {elapsed/len(omega2_grid)*1000:.1f} ms")
print("Compare: L08 PINN / L09 differentiable solver would re-run a full optimization per observation.")


**The amortization win, stated plainly.** A PINN (L08) or differentiable-physics solver
(L09) would launch a *fresh* gradient-descent optimization — potentially seconds to minutes —
for **each** of these five observations. The NPE answered all five with a single pre-trained
network in milliseconds. Over a catalog of thousands of measurements (think gravitational-wave
events, a sky survey, repeated lab shots), the one-time training cost is amortized away and the
per-event cost collapses to near zero. **That is why SBI is the method of choice for
large-scale inference in modern physics.**


## 8. Can we trust the error bars? — a coverage check (SBC-lite)

A posterior is only useful if it is **calibrated**: when it says "90% credible interval", the
truth should fall inside 90% of the time. An amortized network can be silently *over-confident*
(too-narrow posteriors) or *under-confident* (too-wide), and the pretty histograms above won't
reveal it. We need a quantitative test.

The standard tool is **Simulation-Based Calibration (SBC)** (Talts et al. 2018). The lite
version, repeated over many ground-truth draws:

1. Draw $\vartheta_{\text{true}} \sim p(\vartheta)$ **from the prior**.
2. Simulate an observation $x \sim \text{simulator}(\vartheta_{\text{true}})$.
3. Draw $L$ posterior samples $\{\vartheta_\ell\} \sim q_\phi(\vartheta\mid x)$.
4. Record the **rank** of $\vartheta_{\text{true}}$ among those samples (how many fall below it).

**The theorem:** if the inferred posterior equals the true posterior, the ranks are
**uniformly distributed**. So we just histogram the ranks and look for flatness.

| rank histogram shape | diagnosis |
|---|---|
| flat / uniform | calibrated — trust the error bars |
| ∪-shaped (peaks at both ends) | **over-confident** — posteriors too narrow |
| ∩-shaped (peak in the middle) | **under-confident** — posteriors too wide |
| sloped / one-sided peak | **biased** — posterior systematically off to one side |

This is the UQ thread the course keeps emphasizing: *a model that quantifies its own
uncertainty must also let you check that the uncertainty is honest.*


### Checkpoint 3 — implement the SBC-lite rank histogram

Complete the `rank` computation in the loop below (one `# TODO` line): the rank is the number
of posterior samples strictly less than $\vartheta_{\text{true}}$. Then read the histogram
and classify the calibration using the table above.


In [ ]:
def sbc_lite(flow, simulate_fn, summary_fn, prior_sampler,
             n_sims=200, n_post=1000, key=jr.PRNGKey(0)):
    '''Return an array of posterior ranks of theta_true (one per simulation).'''
    ranks = []
    for i in range(n_sims):
        key, k_th, k_sim, k_post = jr.split(key, 4)
        theta_true = prior_sampler(k_th)                 # scalar omega^2 from the prior
        x = summary_fn(simulate_fn(float(theta_true), k_sim))
        c_n = norm_c(x[None, :])[0]
        post_n = flow.sample_given_one(k_post, c_n, n_post)       # (n_post, 1) standardized
        post   = denorm_theta(post_n)[:, 0]                       # physical units
        # TODO (Checkpoint 3): rank = number of posterior samples below theta_true.
        rank = int(jnp.sum(post < theta_true))
        ranks.append(rank)
    return np.asarray(ranks)


prior_A = lambda k: jr.uniform(k, (), minval=0.5, maxval=2.0)
ranks_A = sbc_lite(npe_A, simulate_oscillator, summary_oscillator, prior_A,
                   n_sims=200, n_post=1000, key=jr.PRNGKey(321))

plt.figure(figsize=(7, 3.4))
plt.hist(ranks_A, bins=20, color="C0", alpha=0.7, edgecolor="white")
plt.axhline(len(ranks_A) / 20, color="C3", ls="--", lw=2, label="uniform (ideal)")
plt.xlabel(r"rank of $\omega^2_{\rm true}$ among posterior samples")
plt.ylabel("count"); plt.title("SBC-lite rank histogram (target A)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

# Quick numeric summary to aid the diagnosis.
lo = np.mean(ranks_A < 1000 * 0.1)     # fraction in lowest 10% of ranks
hi = np.mean(ranks_A > 1000 * 0.9)     # fraction in highest 10% of ranks
print(f"fraction in lowest-10%  ranks: {lo:.2f}  (ideal ~0.10)")
print(f"fraction in highest-10% ranks: {hi:.2f}  (ideal ~0.10)")
print("Flat histogram -> calibrated. U-shape -> over-confident. Hump -> under-confident.")


**How to read your result.** A roughly flat histogram (bars hovering near the red dashed
line) means the NPE is *calibrated* — its 90% intervals really contain the truth ~90% of the
time. A pronounced ∪ shape would warn you the posteriors are too tight (over-confident), and a
central hump would mean they are too loose. With a small training set or too few epochs you will
often see mild over-confidence; growing `N_TRAIN_A` or training longer flattens it. **Never ship
an amortized posterior without this check.**


## 9. Target B — infer the temperature of a double-well from its histogram

For a second, qualitatively different target we reuse the **L15 double-well** energy. A particle
diffusing in

$$
U(x_1, x_2) = (x_1^2 - 1)^2 + \tfrac{1}{2}x_2^2
$$

at temperature $k_BT$ samples the Boltzmann distribution $\propto e^{-U/k_BT}$. The **barrier**
between the two wells at $x_1 = \pm 1$ has height $\approx 1$. The unknown physical parameter is
the **temperature** $\vartheta \equiv k_BT$, with prior $k_BT \sim \mathcal{U}[0.1, 0.5]$:

- **Low $k_BT$** → the particle stays trapped in one well; the histogram is a single sharp peak.
- **High $k_BT$** → the particle hops the barrier; the histogram is bimodal and broad.

So the *shape of the position histogram* encodes the temperature — a textbook inverse problem.
Our observation is a coarse **histogram of $x_1$** (8 bins) from a Metropolis chain. The flow
will infer $p(k_BT \mid \text{histogram})$.


In [ ]:
# ── L15 double-well energy + Metropolis sampler (reused verbatim) ──
def double_well(x, kT=0.2):
    # Returns U(x)/kT  (so we never divide later).
    return ((x[..., 0]**2 - 1)**2 + 0.5 * x[..., 1]**2) / kT


@partial(jax.jit, static_argnames=("energy_fn", "n_steps", "step_size"))
def metropolis_mcmc(energy_fn, x0, kT, n_steps, step_size, key):
    def step(carry, k):
        x, n_acc = carry
        k1, k2 = jr.split(k)
        x_prop = x + step_size * jr.normal(k1, x.shape)
        dE = energy_fn(x_prop, kT) - energy_fn(x, kT)
        accept = jr.uniform(k2) < jnp.exp(-dE)
        x_new = jnp.where(accept, x_prop, x)
        return (x_new, n_acc + accept.astype(jnp.int32)), x_new
    keys = jr.split(key, n_steps)
    (_, n_acc), traj = jax.lax.scan(step, (x0, jnp.int32(0)), keys)
    return traj, n_acc


HIST_BINS = 8
HIST_EDGES = jnp.linspace(-2.2, 2.2, HIST_BINS + 1)
C_DIM_B = HIST_BINS

def simulate_doublewell_hist(kT, key, n_steps=4000, step_size=0.25):
    '''Run a Metropolis chain at temperature kT; return a normalized 8-bin histogram of x1.'''
    x0 = jnp.array([-1.0, 0.0])
    traj, _ = metropolis_mcmc(double_well, x0, float(kT), n_steps, step_size, key)
    counts, _ = jnp.histogram(traj[:, 0], bins=HIST_EDGES)
    return counts / (jnp.sum(counts) + 1e-8)        # (8,) normalized

# Show how the histogram changes with temperature.
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
centers = 0.5 * (np.asarray(HIST_EDGES)[:-1] + np.asarray(HIST_EDGES)[1:])
for ax, kT in zip(axes, [0.12, 0.25, 0.45]):
    h = np.asarray(simulate_doublewell_hist(kT, jr.PRNGKey(int(kT*100))))
    ax.bar(centers, h, width=0.45, color="C1", alpha=0.8)
    ax.set_title(f"$k_BT = {kT}$"); ax.set_xlabel("$x_1$"); ax.set_ylim(0, 0.5)
    ax.grid(alpha=0.3)
axes[0].set_ylabel("fraction")
plt.suptitle("Forward simulator B: double-well $x_1$ histogram vs temperature", y=1.04)
plt.tight_layout(); plt.show()
print(f"observation dimension (c_dim) = {C_DIM_B}")


In [ ]:
# Build the target-B training set: kT ~ U[0.1, 0.5] -> histogram.
def make_dataset_B(n, key):
    k_theta, k_sim = jr.split(key)
    kT = jr.uniform(k_theta, (n,), minval=0.1, maxval=0.5)
    sim_keys = jr.split(k_sim, n)
    # Metropolis loop is not vmap-friendly here (python histogram); use a comprehension.
    C = []
    for i in range(n):
        C.append(simulate_doublewell_hist(float(kT[i]), sim_keys[i]))
    C = jnp.stack(C)
    return kT[:, None], C

# Smaller set than target A: each simulation runs a 4000-step chain.
N_TRAIN_B = 4000
theta_B, c_B = make_dataset_B(N_TRAIN_B, jr.PRNGKey(11))
print(f"theta_B shape: {theta_B.shape},  c_B shape: {c_B.shape}")

# Reuse-named standardizers for target B (distinct objects).
theta_mean_B, theta_std_B = theta_B.mean(0), theta_B.std(0)
c_mean_B, c_std_B = c_B.mean(0), c_B.std(0) + 1e-6
def norm_theta_B(t): return (t - theta_mean_B) / theta_std_B
def denorm_theta_B(t): return t * theta_std_B + theta_mean_B
def norm_c_B(c): return (c - c_mean_B) / c_std_B

theta_B_n, c_B_n = norm_theta_B(theta_B), norm_c_B(c_B)


In [ ]:
# Train an NPE for target B. We can reuse train_npe but it standardizes via the global
# norm_c used inside sample_given_one only for sampling; here log_prob works on already-
# standardized inputs, so training is fine. (Sampling uses the flow directly.)
npe_B = ConditionalRealNVP(d=1, c_dim=C_DIM_B, n_layers=10, hidden=64, rngs=nnx.Rngs(1))
losses_B = train_npe(npe_B, theta_B_n, c_B_n, n_epochs=400, batch_size=256, lr=1e-3)

plt.figure(figsize=(7, 3))
plt.plot(losses_B); plt.xlabel("epoch"); plt.ylabel("NLL")
plt.title("NPE training loss (target B: double well)"); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# Posterior for a held-out temperature kT_true = 0.30.
kT_true = 0.30
h_obs   = simulate_doublewell_hist(kT_true, jr.PRNGKey(777))        # (8,)
c_obs_B = norm_c_B(h_obs[None, :])[0]
post_B  = np.asarray(denorm_theta_B(
              npe_B.sample_given_one(jr.PRNGKey(88), c_obs_B, 4000)))[:, 0]

prior_std_B = (0.5 - 0.1) / np.sqrt(12)
print(f"posterior mean kT = {post_B.mean():.3f}  (true = {kT_true})")
print(f"posterior std  kT = {post_B.std():.3f}    (prior std = {prior_std_B:.3f})")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].bar(centers, np.asarray(h_obs), width=0.45, color="C1", alpha=0.8)
axes[0].set_title(f"observation: histogram at $k_BT={kT_true}$"); axes[0].set_xlabel("$x_1$")
axes[1].hist(post_B, bins=50, density=True, color="C0", alpha=0.65,
             label="posterior $p(k_BT|\\,\\rm hist)$")
axes[1].axvspan(0.1, 0.5, color="gray", alpha=0.12, label="prior")
axes[1].axvline(kT_true, color="C3", lw=2, label="true $k_BT$")
axes[1].set_xlabel("$k_BT$"); axes[1].set_title("inferred posterior"); axes[1].legend(fontsize=8)
for ax in axes: ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


**Same machinery, different physics.** With *zero* changes to the flow class — only a new
simulator and a new summary statistic (an 8-bin histogram instead of a 4-vector) — we inferred a
temperature from barrier-crossing statistics. This is the modularity that makes SBI attractive:
the inference engine is decoupled from the physics. Swap the simulator, keep the flow.


## 10. From toy to frontier — where SBI is doing real physics

Our two simulators are deliberately tiny so everything runs in minutes on a CPU. But the
*exact* method — train a conditional density estimator on forward simulations, then amortize —
is now standard in production physics pipelines. A one-slide survey:

**Gravitational waves (LIGO/Virgo).** Estimating the parameters of a binary merger (masses,
spins, sky location) traditionally needs hours of MCMC *per event* against an expensive
waveform likelihood. The **DINGO** method (Dax et al., *Phys. Rev. Lett.* 2021) trains a
conditional normalizing flow on simulated waveforms and produces a full posterior over all
15 parameters in **seconds**, with results that match the gold-standard samplers. With a
catalog of dozens of events and growing, amortization is decisive.

**Cosmology.** Inferring cosmological parameters ($\Omega_m$, $\sigma_8$, ...) from a survey
involves forward models (N-body simulations, hydro) whose likelihood is wholly intractable.
SBI lets cosmologists work directly with simulation outputs (e.g. the *SimBIG* program for
galaxy clustering), bypassing hand-built summary statistics and Gaussian-likelihood
assumptions.

**Particle physics, astro, neuroscience.** The same template appears in LHC measurements,
exoplanet atmosphere retrieval, and computational neuroscience. The community tool is the
open-source **`sbi`** package (Tejero-Cantero et al., *JOSS* 2020), which implements **SNPE-C**
(Greenberg et al., 2019) and its siblings — the same Neural Posterior Estimation you just built,
plus sequential refinement and likelihood/ratio variants.

> **The throughline.** Today's `ConditionalRealNVP` is a teaching-scale version of exactly
> these tools. The leap from this notebook to a gravitational-wave pipeline is *more
> simulations and a bigger flow* — not a different idea.


In [ ]:
# OPTIONAL (Colab, needs `pip install sbi` + PyTorch): a 3-line SNPE-C on our oscillator.
# This mirrors what we built by hand, using the production library. Skip if sbi isn't installed.
RUN_SBI_DEMO = False   # flip to True in Colab after installing sbi

if RUN_SBI_DEMO:
    import torch
    from sbi.inference import SNPE
    from sbi.utils import BoxUniform

    prior = BoxUniform(low=torch.tensor([0.5]), high=torch.tensor([2.0]))

    def sim_torch(theta):
        # theta: (batch, 1) torch -> summary (batch, 4) torch
        out = []
        for o2 in theta[:, 0].tolist():
            x = simulate_oscillator(float(o2), jr.PRNGKey(np.random.randint(1 << 30)))
            out.append(np.asarray(summary_oscillator(x)))
        return torch.tensor(np.stack(out), dtype=torch.float32)

    theta_t = prior.sample((3000,))
    x_t = sim_torch(theta_t)
    inference = SNPE(prior=prior)
    inference.append_simulations(theta_t, x_t).train()
    posterior = inference.build_posterior()

    x_o = summary_oscillator(simulate_oscillator(1.0, jr.PRNGKey(123)))
    samples = posterior.sample((3000,), x=torch.tensor(np.asarray(x_o), dtype=torch.float32))
    print("sbi SNPE-C posterior mean:", float(samples.mean()))
else:
    print("sbi demo skipped. The hand-built ConditionalRealNVP above is the same method.")


## 11. Summary

| Concept | One-line takeaway |
|---|---|
| Inverse problem | data → parameters; the honest answer is a *posterior*, not a point |
| Likelihood-free | for a black-box simulator we can sample $x\mid\vartheta$ but not evaluate its density |
| Conditional flow | L15 RealNVP + concatenate the observation $c$ into the coupling nets — that's the whole change |
| NPE | train the conditional flow on forward-simulated $(\vartheta, c)$ pairs with NLL → it learns $p(\vartheta\mid c)$ |
| Amortization | train once, then *instant* posteriors for any new observation (vs L08/L09 re-optimizing each time) |
| Native UQ | the posterior width *is* the error bar — and SBC-lite lets you check it is honest |

**The single most important sentence of the lecture:** *We turned a generative model (L15)
into an inference engine by letting its neural networks read the data — one concatenation,
and a flow becomes a Bayesian posterior estimator.*

### What we reused, and what was new

- **Reused from L15:** `AffineCoupling`/`RealNVP` structure, the maximum-likelihood (NLL) loss, the `double_well` energy and Metropolis sampler.
- **New today:** the `c`-concatenation (`ConditionalAffineCoupling`), two physics simulators + summary statistics, the conditional NPE loss/training loop, the amortized-inference demo, and the SBC-lite coverage check.

### Connections across the course

- **L08 (PINN inverse) / L09 (differentiable physics):** gradient-based, point-estimate, per-observation. NPE is likelihood-free, distributional, amortized. Same goal, complementary trade-offs (Section 2).
- **L15 (normalizing flow):** the literal engine — we extended that code, we did not replace it.
- **L17–18 (score / flow matching):** modern SBI increasingly uses these continuous-time generative models as the conditional density estimator instead of RealNVP. The recipe (train on forward simulations, condition on the observation) is identical.


## 12. Exercises

**Checkpoint 1 — Conditional coupling layer** *(done above).* Confirm `log_prob(theta, c)` has
shape `(batch,)` and that sweeping `c` with `theta` fixed moves `log_prob` non-monotonically.

**Checkpoint 2 — NPE training loop** *(done above).* After 500 epochs, verify the posterior mean
for $\omega^2_{\text{true}}=1.0$ lands within $\pm 0.15$ and the posterior std is below the prior std.

**Checkpoint 3 — Posterior coverage (SBC-lite)** *(done above).* Build the rank histogram over
≥100 prior draws and classify the calibration (flat / ∪ / ∩ / sloped).

### Going further (optional)

1. **Sharper posteriors.** Increase `N_TRAIN_A` (8000 → 20000) and/or `n_layers`. Re-run the SBC-lite
   check: does over-confidence shrink? Plot posterior std vs training-set size.

2. **Worse summaries.** Replace `summary_oscillator` with *just* `jnp.var(traj)` (a 1-D summary).
   The posterior should widen — you discarded information about $\omega^2$. This makes concrete why
   *summary-statistic choice* is the crux of practical SBI.

3. **Misspecification.** Train target A with measurement `noise=0.05`, then evaluate on an observation
   simulated with `noise=0.30`. Does the posterior still cover the truth? (It often won't — model
   misspecification is SBI's main real-world failure mode; SBC-lite will flag it.)

4. **Two parameters at once.** Extend target A to infer *both* $\omega^2$ and the noise level
   (`d=2`). You'll see a 2-D posterior with correlations between the parameters — plot it as a
   2-D histogram and interpret the degeneracy physically.

5. **Production check.** In Colab, `pip install sbi`, set `RUN_SBI_DEMO = True`, and compare the
   library's SNPE-C posterior to your hand-built flow on the same oscillator observation.

## References

1. Papamakarios & Murray, *Fast ε-free Inference of Simulation Models with Bayesian Conditional Density Estimation*, NeurIPS (2016).
2. Greenberg, Nonnenmacher & Macke, *Automatic Posterior Transformation for Likelihood-Free Inference* (SNPE-C), ICML (2019).
3. Cranmer, Brehmer & Louppe, *The frontier of simulation-based inference*, PNAS **117**, 30055 (2020).
4. Talts, Betancourt, Simpson, Vehtari & Gelman, *Validating Bayesian Inference Algorithms with Simulation-Based Calibration*, arXiv:1804.06788 (2018).
5. Dax et al., *Real-Time Gravitational Wave Science with Neural Posterior Estimation* (DINGO), *Phys. Rev. Lett.* **127**, 241103 (2021).
6. Tejero-Cantero et al., *sbi: A toolkit for simulation-based inference*, *J. Open Source Softw.* **5**, 2505 (2020).
7. Dinh, Sohl-Dickstein & Bengio, *Density Estimation using Real NVP*, ICLR (2017).
